# Task 3 — Hybrid Unlearning v2 (Merged SOTA Pipeline)

This notebook runs the **Hybrid v2** unlearning pipeline built from our `diagnostics.json` forensics and the public leaderboard findings (`#2120` = `0.0956`, `T3-011` in-domain retain):

1. **Heavy-tail margin + patch fragility oracle** (`margin_power=-2.2..-3.0`, `fragility_boost`, no `soften`, no `view_ratio`).
2. **Surgical 55-class frozen head** (`build_active_head_mask` keeps all 55 non-forget class rows exact to `model.pt`) + `blocks.6..11` focus (`ls2` memorization site).
3. **In-domain 45-class retain anchors** (`gen_indomain.py` with SDXL-Turbo) mixed with COCO `val2017` + split **Active-45 vs Inactive-55** retain audits.
4. **5 calibrated branches + Layerwise Soup (`head gamma <= 1.0`) + uniform gamma sweep** (`1.0, 1.2, 1.35, 1.5`).

**Runtime guide:** A100 (~10–12 min, `MODE="full"`) · L4 (~18 min, `MODE="full"`) · T4 (~28 min, `MODE="fast"`).

In [ ]:
import os
import subprocess
from pathlib import Path

import torch

IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# --- Configuration (override MODE if desired: "full" or "fast") ---
TASK_DIR = Path("/content/drive/MyDrive/task3") if IN_COLAB else Path.cwd() / "task3"
OUT_DIR = TASK_DIR / "output" / "hybrid_v2"
REPO_DIR = Path("/content/argos-ai-european-championship") if IN_COLAB else Path.cwd()
REPO_URL = "https://github.com/Iliane-c/argos-ai-european-championship.git"
BRANCH = "t3/task3-unlearning-state-review"
FORCE_RERUN = False

assert torch.cuda.is_available(), "Switch Colab Runtime -> Change runtime type -> GPU (A100/L4/T4)!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
MODE = "full" if vram_gb >= 20.0 else "fast"
AMP = "bf16" if torch.cuda.is_bf16_supported() else "fp16"

print(f"GPU      : {gpu_name} ({vram_gb:.1f} GB VRAM), bf16={torch.cuda.is_bf16_supported()}")
print(f"MODE/AMP : {MODE} / {AMP}")
print(f"TASK_DIR : {TASK_DIR}")
print(f"OUT_DIR  : {OUT_DIR}")
subprocess.run(["nvidia-smi", "-L"], check=False)

In [ ]:
import getpass
import subprocess
import sys

def run_cmd(cmd, cwd=None, check=True):
    print("+", " ".join(str(c) for c in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check)

if not (REPO_DIR / ".git").exists():
    try:
        run_cmd(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])
    except subprocess.CalledProcessError:
        gh_token = getpass.getpass("Repo appears private — enter GitHub PAT (input hidden): ").strip()
        authed_url = REPO_URL.replace("https://", f"https://x-access-token:{gh_token}@")
        run_cmd(["git", "clone", "--branch", BRANCH, authed_url, str(REPO_DIR)])
else:
    run_cmd(["git", "fetch", "origin"], cwd=REPO_DIR)
    run_cmd(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run_cmd(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)

run_cmd([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "timm", "diffusers", "transformers", "accelerate", "huggingface_hub", "requests", "matplotlib",
])
run_cmd(["git", "log", "-n", "3", "--oneline"], cwd=REPO_DIR)

In [ ]:
import os
from pathlib import Path

from huggingface_hub import hf_hub_download

TASK_DIR.mkdir(parents=True, exist_ok=True)
for fname in ("model.pt", "forget_images.npz", "task_template.py"):
    dest = TASK_DIR / fname
    if not dest.exists():
        print(f"Downloading {fname} from SprintML/unlearning ...")
        hf_hub_download(
            repo_id="SprintML/unlearning",
            repo_type="dataset",
            filename=fname,
            local_dir=str(TASK_DIR),
        )
    print(f"[ok] {dest} ({dest.stat().st_size / (1024 ** 2):.1f} MB)")

# Reuse existing COCO pool if v1 already downloaded it so we don't re-fetch 778 MB
POOL_CANDIDATES = [
    TASK_DIR / "coco224.npy",
    OUT_DIR / "pools" / "val2017",
    TASK_DIR / "output" / "hybrid" / "pools" / "val2017",
]
DETECTED_POOL = None
for cand in POOL_CANDIDATES:
    if cand.is_file() or (cand.is_dir() and any(cand.glob("*.jpg"))):
        DETECTED_POOL = cand
        os.environ["POOL_PATH"] = str(cand)
        break

print("Detected COCO pool:", DETECTED_POOL if DETECTED_POOL else "None (pipeline will auto-download COCO val2017)")

In [ ]:
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

INDOMAIN_PER_CLASS = 120 if MODE == "full" else 60
INDOMAIN_PATH = OUT_DIR / "pools" / "gen_indomain.npy"
EXISTING_INDOMAIN = next(
    (p for p in (INDOMAIN_PATH, TASK_DIR / "gen_indomain.npy", TASK_DIR / "output" / "hybrid" / "pools" / "gen_indomain.npy") if p.is_file()),
    INDOMAIN_PATH,
)
INDOMAIN_PATH = EXISTING_INDOMAIN
os.environ["INDOMAIN_FILE"] = str(INDOMAIN_PATH)

if FORCE_RERUN or not INDOMAIN_PATH.is_file():
    print(f"Generating 45-class in-domain retain images ({INDOMAIN_PER_CLASS}/class) -> {INDOMAIN_PATH}")
    subprocess.run(
        [
            sys.executable, "-u", str(REPO_DIR / "unlearning" / "gen_indomain.py"),
            "--out", str(INDOMAIN_PATH),
            "--per-class", str(INDOMAIN_PER_CLASS),
            "--batch", "8",
        ],
        check=True,
    )
else:
    print(f"[skip] Using cached in-domain anchors at {INDOMAIN_PATH}")

# Preview 15 generated samples across classes
imgs_gen = np.load(INDOMAIN_PATH, mmap_mode="r")
lbl_path = INDOMAIN_PATH.with_suffix(".labels.npy")
lbls_gen = np.load(lbl_path) if lbl_path.is_file() else None
print(f"gen_indomain shape: {imgs_gen.shape}, dtype={imgs_gen.dtype}")

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(imgs_gen), size=min(15, len(imgs_gen)), replace=False)
fig, axes = plt.subplots(3, 5, figsize=(13, 8))
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(imgs_gen[idx])
    title = f"cls {int(lbls_gen[idx])}" if lbls_gen is not None else f"idx {idx}"
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import os
import subprocess

env = os.environ.copy()
env.update({
    "TASK_DIR": str(TASK_DIR),
    "OUT_DIR": str(OUT_DIR),
    "MODE": MODE,
    "STAGES": "retain,oracle,wave1,wave2,soup",
    "USE_INDOMAIN": "1",
    "INDOMAIN_FILE": str(INDOMAIN_PATH),
    "SKIP_SHADOW": "1",
    "FORCE": "1" if FORCE_RERUN else "0",
    "AUTO_INSTALL_DEPS": "1",
    "AMP": AMP,
})
if DETECTED_POOL is not None:
    env["POOL_PATH"] = str(DETECTED_POOL)

proc = subprocess.Popen(
    ["bash", str(REPO_DIR / "unlearning" / "run_pipeline.sh")],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"run_pipeline.sh failed with exit code {rc}")

In [ ]:
import json

def load_json(rel):
    p = OUT_DIR / rel
    return json.loads(p.read_text()) if p.is_file() else None

ret_meta = load_json("retain/retain_meta.json")
ora_meta = load_json("oracle/oracle_targets.json")
soup_meta = load_json("final/soup_metrics.json")

if ret_meta:
    print(
        f"Retain proxy : fit={ret_meta['fit_count']}, audit={ret_meta['audit_count']} "
        f"(active45={ret_meta.get('audit_active_count')}, inactive55={ret_meta.get('audit_inactive_count')}), "
        f"active_classes={ret_meta['active_classes']}/100"
    )
if ora_meta:
    print(
        f"Oracle ({ora_meta.get('delta_mode')}): mp={ora_meta.get('margin_power')}, "
        f"delta p10/med/p90={ora_meta.get('delta_p10', 0):.1f}/{ora_meta.get('delta_median', 0):.1f}/{ora_meta.get('delta_p90', 0):.1f}, "
        f"target CE={ora_meta['target_ce_mean']:.3f}+/-{ora_meta['target_ce_std']:.3f}"
    )

rows = []
branch_files = [
    ("wave1_last6", "wave1_last6/last6_mp22.json"),
    ("wave1_last5_patch", "wave1_last5_patch/last5_patch_mp25.json"),
    ("wave2_full_mp30", "wave2_branches/branch_full_mp30.json"),
    ("wave2_npo_moment", "wave2_branches/branch_hybrid_npo_moment.json"),
]
for name, rel in branch_files:
    d = load_json(rel)
    if d:
        rows.append((name, d))
kf = load_json("wave2_kfold/kfold_metrics.json")
if kf and kf.get("alpha_sweep"):
    best_kf = max(kf["alpha_sweep"], key=lambda x: x.get("selection_score", -1.0))
    rows.append((f"wave2_kfold(a={best_kf['alpha']})", best_kf))
if soup_meta:
    rows.append(("FINAL_SOUP", soup_meta))
    for sw in soup_meta.get("uniform_gamma_sweep", []):
        rows.append((f"sweep_g{sw['uniform_gamma']:.2f}", sw))

print("\n" + f"{'Model':26s} | {'Clean CE (u/s)':15s} | {'FgtAcc':6s} | {'RetAcc':6s} | {'RetKL':7s} | {'Act45Acc':8s} | {'Inact55Acc':10s}")
print("-" * 95)
print(f"{'REF #2120 (0.0956)':26s} | {'1.600 / 1.550':15s} | {'  --  ':6s} | {'  --  ':6s} | {'  --   ':7s} | {'   --   ':8s} | {'    --    ':10s}")
for label, m in rows:
    ce_str = f"{m.get('forget_clean_ce_mean', 0):.3f} / {m.get('forget_clean_ce_std', 0):.3f}"
    f_acc = f"{m.get('forget_clean_acc', 0):.3f}"
    r_acc = f"{m.get('retain_acc_sub', float('nan')):.3f}"
    r_kl = f"{m.get('retain_mean_kl', float('nan')):.4f}"
    a45 = f"{m.get('active45_retain_acc', float('nan')):.3f}"
    i55 = f"{m.get('inactive55_retain_acc', float('nan')):.3f}"
    print(f"{label:26s} | {ce_str:15s} | {f_acc:6s} | {r_acc:6s} | {r_kl:7s} | {a45:8s} | {i55:10s}")

if soup_meta:
    print("\nSoup weights:", [round(w, 4) for w in soup_meta.get("soup_weights", [])])
    print("Group gammas:", soup_meta.get("group_gammas", {}))
    compact = {
        k: soup_meta.get(k)
        for k in (
            "forget_clean_ce_mean", "forget_clean_ce_std", "forget_clean_acc",
            "forget_alt_ce_mean", "forget_alt_ce_std", "forget_alt_acc",
            "retain_score", "retain_acc_sub", "retain_mean_kl",
            "active45_retain_acc", "active45_retain_kl",
            "inactive55_retain_acc", "inactive55_retain_kl",
            "soup_weights", "group_gammas",
        )
    }
    print("\n=== PASTE BACK TO ASSISTANT ===")
    print(json.dumps(compact, indent=2))

In [ ]:
import getpass
import sys
from pathlib import Path

import requests
import torch

SUBMIT_NOW = False  # Set True when ready to upload to leaderboard
CANDIDATE = OUT_DIR / "final" / "soup_submission.pt"  # Or e.g. soup_submission_uniform_g1.35.pt
SERVER_URL = "http://34.147.229.55/submit/31-unlearning"

assert CANDIDATE.is_file(), f"Missing {CANDIDATE}"
sd = torch.load(CANDIDATE, map_location="cpu", weights_only=True)
assert all(v.dtype == torch.float32 and torch.isfinite(v).all() for v in sd.values())

sys.path.insert(0, str(TASK_DIR))
import task_template as tt
m = tt.build_model()
m.load_state_dict(sd, strict=True)
print(f"Validated {CANDIDATE.name}: {len(sd)} float32 tensors, {CANDIDATE.stat().st_size:,} bytes")

if SUBMIT_NOW:
    api_key = getpass.getpass("Enter Team API Key (hidden): ").strip()
    with CANDIDATE.open("rb") as f:
        resp = requests.post(
            SERVER_URL,
            headers={"X-API-Key": api_key},
            files={"file": (CANDIDATE.name, f)},
            timeout=(30, 600),
        )
    print("HTTP", resp.status_code, resp.text)
else:
    print("Set SUBMIT_NOW = True to upload directly from Colab, or uncomment files.download below:")
    # if IN_COLAB:
    #     from google.colab import files
    #     files.download(str(CANDIDATE))